# Antes de implementar RAG
Ejemplo de como responde el modelo antes de implementar la arquitectura RAG

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Inicializar el modelo
model = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    temperature = 0,
)

# Le decimos al modelo lo que queremos que haga
messages = [
    (
        "system",
        """Eres un asesor financiero que responde preguntas de los usuarios con explicaciones
        adecuadas y concisas para quienes no estan acostumbrados al lenguaje de las finanzas. 
        Tus respuestas deben estar en el idioma en el que fue hecha la pregunta""",
    ),
    ("human", "Segun Aharon L. Lechter que consejo malo dan los padres a sus hijos hoy en dia?"),
]

# Imprimir la respuesta del modelo
msg = model.invoke(messages)
print(msg.content)

Según Aharon L. Lechter, un consejo financiero que los padres suelen dar a sus hijos y que puede ser contraproducente hoy en día es:

**"Ahorra todo lo que puedas y no gastes en cosas que no necesitas."**

**¿Por qué puede ser un mal consejo hoy en día?**

Si bien el ahorro es importante, este consejo, dicho de forma muy estricta, puede llevar a que los hijos:

*   **Se pierdan oportunidades de crecimiento:** Invertir en educación, en un negocio propio, o incluso en experiencias que amplíen sus horizontes, puede ser más beneficioso a largo plazo que simplemente acumular dinero sin usarlo.
*   **Tengan miedo a gastar:** Pueden desarrollar una mentalidad de escasez, donde gastar dinero, incluso en cosas que les traerían felicidad o mejorarían su calidad de vida, se sienta como un error.
*   **No aprendan a usar el dinero de forma inteligente:** Ahorrar es solo una parte de la salud financiera. Aprender a invertir, a gastar de forma consciente y a gestionar deudas (si es necesario) son ha

# Implementacion RAG

### Document loading

In [4]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

path = Path("./db/books")
allDocs = []
# Iteramos cada archivo en la ruta
for file in path.iterdir():
    loader = PyPDFLoader(file)
    pages = loader.load()
    # Interceptamos el archivo para modificar su metadata
    for page in pages:
        if file.name == "Padre_Pobre_Padre_Rico.pdf":
            page.metadata['title'] = 'Padre Rico, Padre Pobre'
            page.metadata['author'] = 'Robert T. Kiyosaki, Sharon L. Lechter' 
            page.metadata['subject'] = 'Finanzas Personales'
    # print(pages[0].metadata)
    allDocs.extend(pages)
    print(f"File {file.name} loaded. Pages: {len(pages)}")

File EL INVERSOR INTELIGENTE - BENJAMIN GRAHAM.pdf loaded. Pages: 1044
File el-pequeno-libro-para-invertir-john-c-bogle.pdf loaded. Pages: 165
File El_hombre_mas_rico_de_Babilonia_George_S_Clason.pdf loaded. Pages: 121
File Los secretos de la mente millonaria - T. Harv Eker.pdf loaded. Pages: 140
File Padre_Pobre_Padre_Rico.pdf loaded. Pages: 231
File Pequeño Cerdo Capitalista.pdf loaded. Pages: 150
File The-Psychology-of-Money-Morgan-Housel.pdf loaded. Pages: 242


### Document splitting

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunkSize = 1200
chunkOverlap = 200

rSplitter = RecursiveCharacterTextSplitter(
    chunk_size = chunkSize,
    chunk_overlap = chunkOverlap
)

splittedDocs = rSplitter.split_documents(allDocs)
print(f"Páginas originales: {len(allDocs)}")
print(f"Chunks generados listos para la base de datos: {len(splittedDocs)}")

Páginas originales: 2093
Chunks generados listos para la base de datos: 4192


### Embeddings y Vectorstores

#### Prueba para medir el tiempo que tarda el cpu en generar los embeddings

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

# Descarga y carga del modelo Google EmbeddingGemma-300m en la GPU
opciones_modelo = {'device': 'cpu'}

embedding = HuggingFaceEmbeddings(
    model_name="google/embeddinggemma-300m",
    model_kwargs=opciones_modelo
)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

In [ ]:
%%time
from langchain_chroma import Chroma

persistDir = "./db/vectorial-db-cpu"

# Inicializacion de una chroma db
chromaDb = Chroma.from_documents(
    documents=splittedDocs,
    embedding=embedding,
    persist_directory=persistDir
)

print(chromaDb._collection.count())

Finished building Vector Database!
Total collections in DB: 4192
4192
CPU times: total: 2h 57min 20s
Wall time: 22min 41s


#### Generacion de los embeddings con la gpu

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

# Descarga y carga del modelo Google EmbeddingGemma-300m en la GPU
opciones_modelo = {'device': 'cuda'}

embedding = HuggingFaceEmbeddings(
    model_name="google/embeddinggemma-300m",
    model_kwargs=opciones_modelo
)
print("Modelo cargado en la tarjeta gráfica")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Modelo cargado en la tarjeta gráfica


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

# Probamos que tal funciona el modelo de embedding con diferentes idiomas
vectorEs = embedding.embed_query("¿Qué es un fondo indexado?")
vectorEn = embedding.embed_query("What is an index fund?")
vectorGe = embedding.embed_query("Was ist ein Indexfonds?")

print(f"Dimension de los vectores: {len(vectorEs)}")

print(f"Cosine similarity: {cosine_similarity([vectorEs], [vectorEn])[0][0]}")
print(f"Cosine similarity: {cosine_similarity([vectorEs], [vectorGe])[0][0]}")
print(f"Cosine similarity: {cosine_similarity([vectorEn], [vectorGe])[0][0]}")

Dimension de los vectores: 768
Cosine similarity: 0.7704835636858518
Cosine similarity: 0.8467191784612165
Cosine similarity: 0.9260788299084759


In [11]:
%%time
from langchain_chroma import Chroma

persistDir = "./db/vectorial-db"

# Inicializacion de una chroma db
chromaDb = Chroma.from_documents(
    documents=splittedDocs,
    embedding=embedding,
    persist_directory=persistDir
)

print(f"Total de colecciones en la DB: {chromaDb._collection.count()}")

Total de colecciones en la DB: 4192
CPU times: total: 5min 5s
Wall time: 3min 7s
